In [18]:
#1. Librerías.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import ast
import re
import unidecode
from news_daily import NewsDailyAggregator
from sklearn.base import BaseEstimator, TransformerMixin

In [19]:
#2. Constantes.
df_llm_path = "../5-LLMs/openai/pruebas_batch/df_final_limpio.csv"
#df_base_path = "../dataset_var_econom_limpio.csv"
df_base_path = "../dataset_v2_fs_top60_+2.csv"
df_exportacion_path = "./dataset_pre_modelo.csv"
df_llm_var_selection_path = "./dataset_llm_var_selection.csv"

In [20]:
#3. Lectura.
df_llm = pd.read_csv(df_llm_path)
df_base = pd.read_csv(df_base_path)
#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None) 

In [21]:
#4. Me quedo con un registro por día de la respuesta del LLM.
#a. Defino la Clase con las funciones de agregación.
class NewsDailyAggregator(BaseEstimator, TransformerMixin):
    """
    Daily feature aggregator for news-derived LLM features.
    Agrupa por fecha y resume numéricas y booleanas con lógicas específicas.
    """

    def __init__(self, date_col="fecha"):
        self.date_col = date_col
        self.feature_names_out_ = None

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df[self.date_col] = pd.to_datetime(df[self.date_col])

        num_cols = df.select_dtypes(include=["number"]).columns.drop(self.date_col, errors="ignore")
        bool_cols = df.select_dtypes(include=["bool"]).columns

        named_aggs = {}

        # --- Numéricas ---
        for col in num_cols:
            named_aggs[f"{col}_mean"] = pd.NamedAgg(column=col, aggfunc="mean")
            named_aggs[f"{col}_std"] = pd.NamedAgg(column=col, aggfunc="std")
            named_aggs[f"{col}_prop_no_cero"] = pd.NamedAgg(column=col, aggfunc=lambda x: (x != 0).mean())

        # --- Booleanas ---
        for col in bool_cols:
            named_aggs[f"{col}_prop_true"] = pd.NamedAgg(column=col, aggfunc="mean")
            named_aggs[f"{col}_sum_true"] = pd.NamedAgg(column=col, aggfunc="sum")
            #named_aggs[f"{col}_any_true"] = pd.NamedAgg(column=col, aggfunc="max")

        # --- Agregación ---
        df_daily = df.groupby(self.date_col).agg(**named_aggs).reset_index()

        # Guardamos los nombres
        self.feature_names_out_ = df_daily.columns.drop(self.date_col)

        return df_daily

    def get_feature_names_out(self):
        return self.feature_names_out_

    def get_feature_names_out(self):
        return self.feature_names_out_
#b. Realizo la agregación propiamente dicha.
agg = NewsDailyAggregator(date_col="fecha")
df_llm_daily = agg.fit_transform(df_llm)

In [22]:
#5. Agrego lags.
#a. Función.
def add_selected_lags(
    df, 
    date_col="fecha", 
    lags=[1,2,3] # Día anterior y Semana Anterior.
):
    """
    Agrega lags para columnas específicas del DataFrame.
    
    """
    df = df.sort_values(date_col).copy()
    df_lagged = df.copy()

    #i. Columnas LLM (todas excepto fecha).
    cols_llm = [c for c in df.columns if c != date_col]

    #ii. Solo usamos las que efectivamente existan en df.
    cols_target = [c for c in (cols_llm) if c in df.columns]

    #iii. Creamos los lags propiamente dichos.
    for lag in lags:
        lagged = df[cols_target].shift(lag)
        lagged.columns = [f"{c}_lag{lag}" for c in lagged.columns]
        df_lagged = pd.concat([df_lagged, lagged], axis=1)

    return df_lagged


#b. Aplicación propiamente dicha.
df_llm_daily_con_lags = add_selected_lags(df_llm_daily)

#c. Borro los primeros 6 días ya que no habia lags para atrás.
df_llm_daily_con_lags = df_llm_daily_con_lags[df_llm_daily_con_lags["fecha"] >= "2025-01-06"]

In [23]:
#6. Joineo con el dataframe base con variables económicas.
#a. Convierto a datetime.
df_base["fecha"] = pd.to_datetime(df_base["fecha"])
df_llm_daily_con_lags["fecha"] = pd.to_datetime(df_llm_daily_con_lags["fecha"])
#b. Mergeo.
df_final = df_base.merge(df_llm_daily_con_lags, how="inner", on="fecha")

In [24]:
df_final['merval_apertura_+2'] = df_base['merval_apertura_+2']

In [26]:
df_final['merval_apertura_+2']

0     198360.4844
1     201379.7500
2     213794.2500
3     213794.2500
4     215724.7031
         ...     
72    281827.4688
73    290890.5313
74    299421.7500
75    303058.3750
76    304796.9688
Name: merval_apertura_+2, Length: 77, dtype: float64

In [27]:
df_final.to_csv(df_exportacion_path, index=False)

In [11]:
#7. Feature selection.

In [12]:
#8. Dataset final

In [ ]:
# Siguientes pasos.
#a. Entrenamiento.
#b. Predicción.
#c. Comparación modelo base vs modelo con openai.
#d. Conclusiones.